In [ ]:
import os
import torch
from PIL import Image
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    EulerAncestralDiscreteScheduler,
)
from diffusers.utils import make_image_grid

# 1. Configuración general
OUTPUT_DIR = "outputs_products"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Definir modelo base y scheduler
MODEL_ID = "stabilityai/stable-diffusion-2-1-base"

# Pipeline texto->imagen
txt2img_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
txt2img_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    txt2img_pipe.scheduler.config
)
txt2img_pipe = txt2img_pipe.to(DEVICE)

# Pipeline img->img (para variaciones a partir de imagen)
img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
img2img_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    img2img_pipe.scheduler.config
)
img2img_pipe = img2img_pipe.to(DEVICE)


def generate_product_images():
    # 3.1 Producto ficticio: sudadera
    prompt_sneakers = (
        "Fotos de alta definicion de una sudadera con fondo blanco"
    )
    image_sneakers = txt2img_pipe(
        prompt=prompt_sneakers,
        guidance_scale=7.5,
        num_inference_steps=30,
    ).images[0]
    sneakers_path = os.path.join(OUTPUT_DIR, "sudadera_white_bg.png")
    image_sneakers.save(sneakers_path)

    # 4. Variar prompts: otro producto (bolso)
    prompt_bag = (
        "Una gorra con algún detalle dorado, luz de estudio y fondo blanco"
    )
    image_bag = txt2img_pipe(
        prompt=prompt_bag,
        guidance_scale=8.0,
        num_inference_steps=35,
    ).images[0]
    bag_path = os.path.join(OUTPUT_DIR, "gorras.png")
    image_bag.save(bag_path)

    print(f"Guardado: {sneakers_path}")
    print(f"Guardado: {bag_path}")

    return sneakers_path, bag_path


def generate_variations_from_image(base_image_path):
    # 5.1 Cargar imagen base de producto
    init_image = Image.open(base_image_path).convert("RGB")
    init_image = init_image.resize((512, 512))  # tamaño para SD 2.1 base

    # 5.2 Prompt para variación (cambiar color y añadir detalles)
    prompt_variation_1 = (
        "Fotos de alta definicion de una sudadera con los colores azul y dorado, el fondo debe ser blanco",
        "Fotos de alta definicion de una sudadera con los colores roja con capucha, el fondo debe ser blanco"
    )

    # strength: cuánto se aleja la nueva imagen de la original (0-1)
    img_var1 = img2img_pipe(
        prompt=prompt_variation_1,
        image=init_image,
        strength=0.6,
        guidance_scale=8.0,
        num_inference_steps=30,
    ).images[0]

    var1_path = os.path.join(OUTPUT_DIR, "sudadera_azuL_blanca.png")
    var2_path = os.path.join(OUTPUT_DIR, "sudadera_roja_capucha.png")
    img_var1.save(var1_path)
    img_var2.save(var2_path)

    # Grid para comparar original + variaciones
    grid = make_image_grid([init_image, img_var1, img_var2], rows=1, cols=3)
    grid_path = os.path.join(OUTPUT_DIR, "sudaderas_variaciones.png")
    grid.save(grid_path)

    print(f"Guardado: {var1_path}")
    print(f"Guardado: {var2_path}")
    print(f"Guardado grid: {grid_path}")


if __name__ == "__main__":
    # Generar imágenes de producto desde texto
    sneakers_path, _ = generate_product_images()

    # Generar variaciones a partir de la imagen de zapatillas
    generate_variations_from_image(sneakers_path)

,overview,main_genre
0,"In the 22nd century, a paraplegic Marine is di...",Action
1,"Captain Barbossa, long believed to be dead, ha...",Adventure
2,A cryptic message from Bond’s past sends him o...,Action
3,Following the death of District Attorney Harve...,Action
4,"John Carter is a war-weary, former military ca...",Action
